In [ ]:
import numpy as np

rng = np.random.default_rng(42)   # random generator; 42 = seed, results are repeatable
u = rng.random()                  # random number between 0 and 1
print(u)

def pick_bucket(u):
    if u<0.65:
        return "loss"
    elif 0.65<=u<0.90:
        return "small"
    elif 0.90<=u<0.96:
        return "good"
    else:
        return "home_run"

counts = {"loss": 0, "small": 0, "good": 0, "home_run": 0}

for i in range(1000000):
    u = rng.random()
    bucket = pick_bucket(u)
    counts[bucket] +=1


for bucket, n in counts.items():
    print(bucket, n/1000000)   

0.7739560485559633
loss 0.650184
small 0.249636
good 0.060151
home_run 0.040029


In [2]:
WRITE_OFF_SHARE = 0.5   # metà delle startup in perdita va a zero (ipotesi)

def draw_multiple(bucket):
    v = rng.random()                      # secondo numero casuale tra 0 e 1
    if bucket == "loss":
        if rng.random() < WRITE_OFF_SHARE:
            return 0.0                    # fallimento totale
        return v                          # recupera qualcosa tra 0 e 1x
    if bucket == "small":
        lo, hi = 1, 5
    elif bucket == "good":
        lo, hi = 5, 10
    else:
        lo, hi = 10, 100
    return lo * (hi / lo) ** v            # log-uniforme: valori bassi più probabili

for b in ["loss", "small", "good", "home_run"]:
    print(b, round(draw_multiple(b), 2))

loss 0.0
small 1.72
good 7.51
home_run 18.22


In [ ]:
def draw_outcome():
    bucket = pick_bucket(rng.random())    # 1. in che fascia cade
    return draw_multiple(bucket)          # 2. quanto rende

N = 10_000
x = np.array([draw_outcome() for _ in range(N)])   # 10.000 startup in un array

print("media   ", round(x.mean(), 2))
print("mediana ", round(np.median(x), 2))
print("a zero  ", round((x == 0).mean(), 2))

top = np.sort(x)[::-1]                    # ordina dalla più grande alla più piccola
print("The best 5% is around", round(top[:N // 20].sum() / x.sum(), 2), "of the total")

media    2.75
mediana  0.52
a zero   0.32
The best 5% is around 0.6 of the total


### What this means

The **median** deal returns 0.52x: half of the startups lose about half of the money. The **mean** is 2.75x only because of a few outliers with huge returns. The median ignores them, while the mean is pulled up by them.

In venture capital those outliers are not noise: they *are* the return. The best 5% of deals produce about 60% of all value.

For a fund this means: with few startups it will probably miss the outliers and end up close to the median, losing money. With many startups it is more likely to catch some, and its result moves towards the mean. **How many startups does a fund need?** That is phase 2.